# MiniCPM-o 4.5 Lip-Reading Test

Runs MiniCPM-V 4.5 (INT4 quantized) on 4 silent video clips to test lip-reading.

**Setup:** Runtime → Change runtime type → T4 GPU

**Steps:**
1. Install dependencies
2. Upload your 4 video clips
3. Run inference
4. Download results JSON

In [ ]:
!pip install -q transformers==4.51.0 accelerate bitsandbytes decord torch pillow

In [ ]:
# Verify GPU is available
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Upload your 4 clip files
from google.colab import files
uploaded = files.upload()  # Select clip_1.mp4 through clip_4.mp4

In [ ]:
# Load MiniCPM-V 4.5 INT4 (pre-quantized, ~11GB VRAM)
from transformers import AutoModel, AutoTokenizer

model_name = "openbmb/MiniCPM-V-4_5-int4"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
import json
from datetime import datetime, timezone
from decord import VideoReader
from PIL import Image
import numpy as np

CLIPS = [
    ("clip_1", "clip_1.mp4", "What are you doing today?"),
    ("clip_2", "clip_2.mp4", "It's very warm this morning."),
    ("clip_3", "clip_3.mp4", "I'd like to take a vacation soon."),
    ("clip_4", "clip_4.mp4", "Did you feed the dog?"),
]

SYSTEM_PROMPT = (
    "You are analyzing a silent video of a person speaking. "
    "The audio has been removed. Based solely on the visual movement "
    "of the speaker's lips, face, and mouth, infer what they are saying. "
    "Respond with ONLY the words you believe were spoken, nothing else."
)

MAX_FRAMES = 24
FRAME_SIZE = (448, 448)  # ViT patch-aligned (multiple of 14)


def extract_frames(video_path, max_frames=MAX_FRAMES):
    vr = VideoReader(video_path)
    total = len(vr)
    # Sample evenly spaced frames up to max_frames
    indices = np.linspace(0, total - 1, min(max_frames, total), dtype=int).tolist()
    frames = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(f).resize(FRAME_SIZE) for f in frames]


results = []
for clip_id, path, ground_truth in CLIPS:
    print(f"Processing {clip_id}...", end=" ", flush=True)
    result = {
        "model": "MiniCPM-o 4.5",
        "model_id": "openbmb/MiniCPM-V-4_5-int4",
        "clip_id": clip_id,
        "ground_truth": ground_truth,
        "response": None,
        "error": None,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    try:
        frames = extract_frames(path)
        print(f"({len(frames)} frames)", end=" ", flush=True)
        messages = [
            {
                "role": "user",
                "content": frames + [SYSTEM_PROMPT + "\nWhat is this person saying?"],
            }
        ]
        response = model.chat(
            image=None,
            msgs=messages,
            tokenizer=tokenizer,
            sampling=False,
        )
        result["response"] = response
        print(f"→ {response!r}")
    except Exception as e:
        result["error"] = str(e)
        print(f"ERROR: {e}")
    results.append(result)

print("\nDone!")

In [ ]:
# Save results and download
output = {"run_id": "colab-minicpm", "results": results}
with open("minicpm_results.json", "w") as f:
    json.dump(output, f, indent=2)

# Print results
for r in results:
    status = r['response'] or f"ERROR: {r['error']}"
    print(f"{r['clip_id']}: {status}")
    print(f"  Ground truth: {r['ground_truth']}")
    print()

files.download("minicpm_results.json")